In [3]:
# ============================================================
# 3- Selenium Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "../drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


In [5]:
# ============================================================
# 4- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [ ]:
# ============================================================
# Extract existing Eitaa contacts into df_exist
# ============================================================
import time
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

SCROLL_PAUSE = 0.8
STABLE_BOTTOM_LIMIT = 4
MAX_SCROLL_STEPS = 300

contacts = {}  # peer_id -> full_name


def get_contacts_list():
    return driver.find_element(By.CSS_SELECTOR, "ul#contacts")


def get_scroll_metrics():
    """Find the actual scrollable ancestor of the contacts list."""
    contacts_list = get_contacts_list()

    result = driver.execute_script("""
        const list = arguments[0];
        let el = list.parentElement;

        while (el) {
            const style = getComputedStyle(el);

            if (
                (style.overflowY === "auto" ||
                 style.overflowY === "scroll") &&
                el.scrollHeight > el.clientHeight
            ) {
                return {
                    scrollTop: el.scrollTop,
                    clientHeight: el.clientHeight,
                    scrollHeight: el.scrollHeight
                };
            }

            el = el.parentElement;
        }

        return null;
    """, contacts_list)

    if result is None:
        raise RuntimeError("Could not find the scrollable contacts container.")

    return result


def collect_visible_contacts():
    """Collect currently rendered contacts without clicking them."""
    rows = driver.find_elements(
        By.CSS_SELECTOR, "ul#contacts > a.chatlist-chat"
    )

    added = 0

    for row in rows:
        peer_id = row.get_attribute("data-peer-id")

        name_elements = row.find_elements(
            By.CSS_SELECTOR, "span.peer-title"
        )

        if not peer_id or not name_elements:
            continue

        full_name = name_elements[0].text.strip()

        if full_name and peer_id not in contacts:
            contacts[peer_id] = full_name
            added += 1

    return added


# Wait until the contact list is available.
WebDriverWait(driver, 15).until(
    lambda d: len(d.find_elements(
        By.CSS_SELECTOR, "ul#contacts > a.chatlist-chat"
    )) > 0
)

# Start at the top to avoid missing contacts above the current position.
contacts_list = get_contacts_list()

driver.execute_script("""
    const list = arguments[0];
    let el = list.parentElement;

    while (el) {
        const style = getComputedStyle(el);

        if (
            (style.overflowY === "auto" ||
             style.overflowY === "scroll") &&
            el.scrollHeight > el.clientHeight
        ) {
            el.scrollTop = 0;
            return;
        }

        el = el.parentElement;
    }
""", contacts_list)

time.sleep(SCROLL_PAUSE)
collect_visible_contacts()

stable_bottom_steps = 0

for step in range(1, MAX_SCROLL_STEPS + 1):
    before = get_scroll_metrics()

    # Scroll down within the actual contacts container.
    contacts_list = get_contacts_list()

    driver.execute_script("""
        const list = arguments[0];
        let el = list.parentElement;

        while (el) {
            const style = getComputedStyle(el);

            if (
                (style.overflowY === "auto" ||
                 style.overflowY === "scroll") &&
                el.scrollHeight > el.clientHeight
            ) {
                const distance = Math.max(
                    300, Math.floor(el.clientHeight * 0.75)
                );

                el.scrollTop = Math.min(
                    el.scrollTop + distance,
                    el.scrollHeight
                );

                return;
            }

            el = el.parentElement;
        }
    """, contacts_list)

    time.sleep(SCROLL_PAUSE)

    added = collect_visible_contacts()
    after = get_scroll_metrics()

    at_bottom = (
            after["scrollTop"] + after["clientHeight"]
            >= after["scrollHeight"] - 3
    )

    no_growth = (
            after["scrollHeight"] <= before["scrollHeight"]
    )

    if at_bottom and no_growth and added == 0:
        stable_bottom_steps += 1
    else:
        stable_bottom_steps = 0

    print(
        f"Step {step}: "
        f"visible={len(driver.find_elements(By.CSS_SELECTOR, 'ul#contacts > a.chatlist-chat'))}, "
        f"new={added}, "
        f"unique_total={len(contacts)}, "
        f"scrollTop={after['scrollTop']}, "
        f"scrollHeight={after['scrollHeight']}, "
        f"stable_bottom={stable_bottom_steps}/{STABLE_BOTTOM_LIMIT}"
    )

    if stable_bottom_steps >= STABLE_BOTTOM_LIMIT:
        print("\nReached a stable bottom; stopping extraction.")
        break
else:
    print(
        "\nWARNING: Maximum scroll steps reached. "
        "The list may not be fully loaded."
    )

# Build the final DataFrame in first-seen order.
df_exist = pd.DataFrame(
    [
        {"peer_id": peer_id, "full_name": full_name}
        for peer_id, full_name in contacts.items()
    ],
    columns=["peer_id", "full_name"]
)

print("\nExtraction summary")
print("------------------")
print("Unique contacts extracted:", len(df_exist))
print("Unique peer IDs:", df_exist["peer_id"].nunique())
print("DataFrame shape:", df_exist.shape)


In [ ]:
df_exist

In [ ]:
# ============================================================
# df = df - df_exist
# ============================================================
df = df.loc[~df["full_name"].isin(df_exist["full_name"])].copy()